# Notebook 05 — Evaluation

Runs all 10 test scenarios through AgentEvaluator and prints:
- Router accuracy
- Allergy safety rate (must be 100%)
- Reflection catch rate
- Latency stats
- Monthly cost estimate

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from src.orchestrator import GiftConciergeAgent
from src.evaluation import AgentEvaluator
from config.settings import Settings

settings = Settings()
agent = GiftConciergeAgent(settings)
evaluator = AgentEvaluator(agent, settings.TEST_SCENARIOS_PATH)
print(f'Loaded {len(evaluator.scenarios)} test scenarios')

📦 Loading embedding model: all-MiniLM-L6-v2…
✅ Embedding model ready
Loaded 10 test scenarios


In [2]:
print('Running evaluation... (this will make real API calls)')
print('='*60)
metrics = evaluator.run_all_scenarios()
print('='*60)

Running evaluation... (this will make real API calls)
  🧪 Running TC001: Basic product search… ✅ intent=PRODUCT_SEARCH, allergy_safe=True
  🧪 Running TC002: Preference update… ✅ intent=PREFERENCE_UPDATE, allergy_safe=True
  🧪 Running TC003: Delivery logistics check… ✅ intent=DELIVERY_CHECK, allergy_safe=True
  🧪 Running TC004: Allergy reflection trap… ✅ intent=PRODUCT_SEARCH, allergy_safe=True
  🧪 Running TC005: Budget-constrained search… ✅ intent=PRODUCT_SEARCH, allergy_safe=True
  🧪 Running TC006: Repeat order reference… ✅ intent=ORDER_HISTORY, allergy_safe=True
  🧪 Running TC007: Unknown recipient… ✅ intent=PRODUCT_SEARCH, allergy_safe=True
  🧪 Running TC008: Cultural occasion — Avurudu… ✅ intent=PRODUCT_SEARCH, allergy_safe=True
  🧪 Running TC009: Delivery to unavailable zone… ✅ intent=DELIVERY_CHECK, allergy_safe=True
  🧪 Running TC010: Daughter dairy trap… ✅ intent=PRODUCT_SEARCH, allergy_safe=True


In [3]:
# ── Summary metrics ─────────────────────────────────────
print('\n📊 EVALUATION RESULTS')
print('='*50)
print(f'Total scenarios       : {metrics["total_scenarios"]}')
print(f'Router accuracy       : {metrics["router_accuracy_pct"]}  (target: 90%+)')
print(f'Allergy safety rate   : {metrics["allergy_safety_rate_pct"]}  (target: 100%)')
print(f'Allergy traps caught  : {metrics["allergy_traps_caught"]} / {metrics["allergy_trap_scenarios"]}')
print(f'Reflection catch rate : {metrics["reflection_catch_rate_pct"]}')
print(f'Avg latency           : {metrics["avg_latency_ms"]} ms')
print(f'Max latency           : {metrics["max_latency_ms"]} ms')

# Allergy safety gate
if metrics['allergy_safety_rate'] >= 1.0:
    print('\n✅ 100% ALLERGY SAFETY ACHIEVED')
else:
    fails = metrics['total_scenarios'] - int(metrics['allergy_safety_rate'] * metrics['total_scenarios'])
    print(f'\n❌ Allergy safety NOT 100% — {fails} violation(s) slipped through!')


📊 EVALUATION RESULTS
Total scenarios       : 10
Router accuracy       : 100.0%  (target: 90%+)
Allergy safety rate   : 100.0%  (target: 100%)
Allergy traps caught  : 4 / 4
Reflection catch rate : 0.0%
Avg latency           : 5812.0 ms
Max latency           : 8594.9 ms

✅ 100% ALLERGY SAFETY ACHIEVED


In [4]:
# ── Per-scenario table ──────────────────────────────────
print('\n📋 Per-Scenario Results:')
print(f'{"ID":<10} {"Name":<35} {"Intent":<8} {"Allergy":<10} {"ms"}')
print('-'*75)
for s in metrics['per_scenario']:
    intent_ok = '✅' if s['intent_correct'] else '❌'
    allergy_ok = '✅' if s['allergy_safe'] else '❌'
    print(f"{s['id']:<10} {s['name'][:33]:<35} {intent_ok}      {allergy_ok}         {s['latency_ms']:.0f}ms")


📋 Per-Scenario Results:
ID         Name                                Intent   Allergy    ms
---------------------------------------------------------------------------
TC001      Basic product search                ✅      ✅         7347ms
TC002      Preference update                   ✅      ✅         4195ms
TC003      Delivery logistics check            ✅      ✅         3377ms
TC004      Allergy reflection trap             ✅      ✅         6758ms
TC005      Budget-constrained search           ✅      ✅         6862ms
TC006      Repeat order reference              ✅      ✅         3888ms
TC007      Unknown recipient                   ✅      ✅         6570ms
TC008      Cultural occasion — Avurudu         ✅      ✅         7148ms
TC009      Delivery to unavailable zone        ✅      ✅         3380ms
TC010      Daughter dairy trap                 ✅      ✅         8595ms


In [5]:
# ── Cost estimate ───────────────────────────────────────
report = evaluator.generate_report_data()
cost = report['cost_estimate']
print('\n💰 Monthly Estimated Cost (500 users × 10 queries/day × 30 days)')
print('='*50)
print(f'Monthly queries     : {cost["monthly_queries"]:,}')
print(f'Input token cost    : ${cost["input_token_cost_usd"]:.2f}')
print(f'Output token cost   : ${cost["output_token_cost_usd"]:.2f}')
print(f'Qdrant (free tier)  : $0.00')
print(f'Total monthly       : ${cost["total_monthly_usd"]:.2f}')
print(f'Cost per query      : ${cost["cost_per_query_usd"]:.4f}')

# Save report
from pathlib import Path
report_path = Path('../output/evaluation_report.json')
report_path.parent.mkdir(exist_ok=True)
with open(report_path, 'w') as f:
    json.dump(report['metrics_summary'], f, indent=2)
print(f'\n💾 Report saved to {report_path}')


💰 Monthly Estimated Cost (500 users × 10 queries/day × 30 days)
Monthly queries     : 150,000
Input token cost    : $300.00
Output token cost   : $720.00
Qdrant (free tier)  : $0.00
Total monthly       : $1020.00
Cost per query      : $0.0068

💾 Report saved to ..\output\evaluation_report.json
